In [48]:
import pandas as pd
import torch
from scripts.data import download_data
import ast
import matplotlib.pyplot as plt
from torchvision.datasets import ImageFolder
from torchvision.transforms import v2
from torch.utils.data import DataLoader

In [3]:
metadata = pd.read_csv(download_data('') / 'metadata.csv', index_col=0)
metadata['shape'] = metadata['shape'].apply(ast.literal_eval) # changing the shape column to tuples
metadata.head(10)

,image,class,format,mode,shape
0,Cancer (1).jpg,tumor,JPEG,RGB,"(512, 512, 3)"
1,Cancer (1).png,tumor,PNG,L,"(300, 240)"
2,Cancer (1).tif,tumor,TIFF,RGB,"(256, 256, 3)"
3,Cancer (10).jpg,tumor,JPEG,RGB,"(512, 512, 3)"
4,Cancer (10).tif,tumor,TIFF,RGB,"(256, 256, 3)"
5,Cancer (100).jpg,tumor,JPEG,RGB,"(512, 512, 3)"
6,Cancer (1000).jpg,tumor,JPEG,RGB,"(290, 250, 3)"
7,Cancer (1001).jpg,tumor,JPEG,RGB,"(620, 620, 3)"
8,Cancer (1002).JPG,tumor,JPEG,RGB,"(338, 264, 3)"
9,Cancer (1003).jpg,tumor,JPEG,RGB,"(442, 353, 3)"


In [83]:
metadata.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4600 entries, 0 to 4599
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   image   4600 non-null   object
 1   class   4600 non-null   object
 2   format  4600 non-null   object
 3   mode    4600 non-null   object
 4   shape   4600 non-null   object
dtypes: object(5)
memory usage: 215.6+ KB


There are no missing metadata values.

In [84]:
metadata['shape'].value_counts()

shape
(512, 512, 3)    884
(225, 225, 3)    364
(630, 630, 3)    126
(256, 256, 3)    105
(236, 236, 3)     89
                ... 
(201, 236, 3)      1
(222, 233, 3)      1
(294, 236, 3)      1
(244, 235, 3)      1
(454, 442, 4)      1
Name: count, Length: 475, dtype: int64

In [95]:
def get_channels(shape):
    if len(shape) == 3:
        return shape[2]
    else:
        return 1

In [99]:
metadata['width'] = metadata['shape'].apply(lambda s: s[0])
metadata['height'] = metadata['shape'].apply(lambda s: s[1])
metadata['channels'] = metadata['shape'].apply(get_channels)

In [109]:
len(metadata.query('width < 224 & height < 224'))

184

In [105]:
metadata.sort_values(by=['width', 'height'])

,image,class,format,mode,shape,width,height,channels
1502,Cancer (2335).jpg,tumor,JPEG,RGB,"(167, 175, 3)",167,175,3
2413,Cancer (909).jpg,tumor,JPEG,RGB,"(167, 175, 3)",167,175,3
2688,Not Cancer (1154).jpg,normal,JPEG,RGB,"(168, 300, 3)",168,300,3
2752,Not Cancer (1211).jpg,normal,JPEG,RGB,"(168, 300, 3)",168,300,3
2760,Not Cancer (1219).jpg,normal,JPEG,RGB,"(168, 300, 3)",168,300,3
...,...,...,...,...,...,...,...,...
1300,Cancer (2155).jpg,tumor,JPEG,L,"(1427, 1275)",1427,1275,1
1366,Cancer (2213).jpg,tumor,JPEG,L,"(1427, 1275)",1427,1275,1
1386,Cancer (2231).jpg,tumor,JPEG,L,"(1427, 1275)",1427,1275,1
2478,Cancer (968).jpg,tumor,JPEG,L,"(1427, 1275)",1427,1275,1


We can see that the smallest images are 167x175, and there are 184 images under 224x224, so we will need to rescale them.

In [118]:
metadata['class'].value_counts() / len(metadata)

class
tumor     0.546304
normal    0.453696
Name: count, dtype: float64

There is a ~55/45 class distribution of cancer and non-cancer scans.

In [121]:
metadata[['mode', 'channels']].value_counts()

mode  channels
RGB   3           4461
L     1            132
RGBA  4              5
P     1              2
Name: count, dtype: int64

We have a few images that do not have 3 channels for RGB, so we will transform them into RGB images during preprocessing.